# 📊 Notebook 03 — LLM Sentiment Analysis

Explore Claude's analysis results: sentiment distribution, confidence, accuracy vs ground truth, theme detection.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
import sys
sys.path.append('..')

from src.config import PROCESSED_DIR, SENTIMENT_COLORS, ENGAGEMENT_THEMES, SUPPORTED_LANGUAGES

df = pd.read_csv(PROCESSED_DIR / 'analyzed_responses.csv')
print(f'Loaded {len(df)} analyzed responses')
df.head(3)

In [ ]:
# ── Sentiment Distribution ─────────────────────────────────────────────────────
order = ['very_positive', 'positive', 'neutral', 'negative', 'very_negative']
sent_counts = df['llm_sentiment'].value_counts().reindex(order).reset_index()
sent_counts.columns = ['Sentiment', 'Count']

fig = px.bar(sent_counts, x='Sentiment', y='Count',
             color='Sentiment', color_discrete_map=SENTIMENT_COLORS,
             title='Overall Sentiment Distribution')
fig.show()

In [ ]:
# ── Ground Truth vs LLM Prediction ────────────────────────────────────────────
# Only valid when llm_sentiment column exists and sentiment_label is ground truth
if 'sentiment_label' in df.columns:
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt

    # Align to same label space
    valid = df[df['llm_sentiment'].notna() & df['sentiment_label'].notna()].copy()
    print(classification_report(valid['sentiment_label'], valid['llm_sentiment']))

    cm = confusion_matrix(valid['sentiment_label'], valid['llm_sentiment'], labels=order)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=order, yticklabels=order, cmap='Blues')
    plt.title('Confusion Matrix: Ground Truth vs LLM Prediction')
    plt.ylabel('Ground Truth'); plt.xlabel('LLM Prediction')
    plt.tight_layout()
    plt.show()
else:
    print('No ground truth column available (external dataset)')

In [ ]:
# ── Confidence Distribution ────────────────────────────────────────────────────
if 'llm_sentiment_confidence' in df.columns:
    fig = px.histogram(df, x='llm_sentiment_confidence', nbins=20,
                       color='llm_sentiment', color_discrete_map=SENTIMENT_COLORS,
                       title='Confidence Distribution by Sentiment',
                       barmode='overlay', opacity=0.7)
    fig.show()

    print(f"Mean confidence: {df['llm_sentiment_confidence'].mean():.3f}")
    print(f"Low confidence (<0.7): {(df['llm_sentiment_confidence'] < 0.7).sum()} rows")

In [ ]:
# ── Sentiment by Language ──────────────────────────────────────────────────────
lang_sent = df.groupby(['language_code', 'llm_sentiment']).size().reset_index(name='Count')
lang_sent['Language'] = lang_sent['language_code'].map(SUPPORTED_LANGUAGES)

fig = px.bar(lang_sent, x='Language', y='Count', color='llm_sentiment',
             color_discrete_map=SENTIMENT_COLORS, barmode='stack',
             title='Sentiment Distribution per Language')
fig.show()

In [ ]:
# ── Theme Detection Accuracy ───────────────────────────────────────────────────
if 'theme_key' in df.columns and 'llm_primary_theme' in df.columns:
    match = (df['theme_key'] == df['llm_primary_theme']).mean()
    print(f'Theme detection accuracy (exact match): {match*100:.1f}%')

    theme_acc = df.groupby('theme_key').apply(
        lambda g: (g['theme_key'] == g['llm_primary_theme']).mean()
    ).reset_index()
    theme_acc.columns = ['Theme', 'Accuracy']
    theme_acc['Theme'] = theme_acc['Theme'].map(ENGAGEMENT_THEMES)
    theme_acc = theme_acc.sort_values('Accuracy')

    fig = px.bar(theme_acc, x='Accuracy', y='Theme', orientation='h',
                 title='Theme Detection Accuracy per Category',
                 color='Accuracy', color_continuous_scale='RdYlGn')
    fig.show()

In [ ]:
# ── Action Signal Distribution ─────────────────────────────────────────────────
if 'llm_action_signal' in df.columns:
    signal_counts = df['llm_action_signal'].value_counts().reset_index()
    signal_counts.columns = ['Signal', 'Count']

    color_map = {
        'urgent_action': '#E74C3C',
        'monitor': '#F39C12',
        'positive_share': '#2ECC71',
        'no_action': '#AEB6BF',
    }
    fig = px.pie(signal_counts, values='Count', names='Signal',
                 color='Signal', color_discrete_map=color_map,
                 title='Action Signal Distribution')
    fig.show()

    # Sample urgent responses
    print('\nSample URGENT responses:')
    urgent = df[df['llm_action_signal'] == 'urgent_action']
    for _, row in urgent.head(3).iterrows():
        lang = SUPPORTED_LANGUAGES.get(row['language_code'], row['language_code'])
        print(f'[{lang}] {row["response_text"][:120]}')
        print(f'  → Summary: {row.get("llm_summary_en", "")}')
        print()

In [ ]:
# ── Emotion Tag Analysis ───────────────────────────────────────────────────────
if 'llm_emotion_tags' in df.columns:
    from collections import Counter

    all_emotions = []
    for tags in df['llm_emotion_tags'].dropna():
        try:
            all_emotions.extend(json.loads(tags))
        except Exception:
            pass

    emotion_counts = pd.DataFrame(Counter(all_emotions).most_common(20),
                                  columns=['Emotion', 'Count'])

    fig = px.bar(emotion_counts, x='Count', y='Emotion', orientation='h',
                 title='Top 20 Detected Emotions')
    fig.show()